In [ ]:
# ✅ Step 0: 下载并安装 Miniforge 到 /opt/miniforge3
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O miniforge.sh
!bash miniforge.sh -b -p /opt/miniforge3

# ✅ Step 0.5: 更新 PATH（当前 Notebook 会话中有效）
import os
os.environ["PATH"] = "/opt/miniforge3/bin:" + os.environ["PATH"]

In [ ]:
with open("f_vcd_environment.yml", "w") as f:
    f.write("""name: FVCD-net

dependencies:
  - python=3.7
  - imageio==2.4.1
  - scipy==1.2.0
  - scikit-image==0.14.1
  - tensorflow-gpu==1.14.0
  - numpy=1.15.4
  - pip
  - pip:
    - easydict==1.9

channels:
  - anaconda
  - conda-forge
  - simpleitk
""")


In [ ]:
#路径设置
import os, sys
os.chdir("/notebooks/Code/DL_net")
sys.path.append("/notebooks/Code/DL_net")

In [ ]:
# ✅ Step 2: 使用 Miniforge 创建 Conda 环境
# ⚠️ 必须用 `!` 运行 shell 命令（适用于 Jupyter）
!source /opt/miniforge3/bin/activate && \
  /opt/miniforge3/bin/conda env create -f /notebooks/f_vcd_environment.yml

In [ ]:
# ✅ Step 3: 注册为 Jupyter Notebook 内核
!source /opt/miniforge3/bin/activate FVCD-net && \
  /opt/miniforge3/envs/FVCD-net/bin/pip install ipykernel && \
  /opt/miniforge3/envs/FVCD-net/bin/python -m ipykernel install --user --name FVCD-net --display-name "Python 3.7 (FVCD)"

In [ ]:
import easydict
print("✅ EasyDict installed from:", easydict.__file__)

In [ ]:
# ✅ 激活 FVCD-net 环境（无需 conda init）
!source /opt/miniforge3/bin/activate FVCD-net && \
  echo "✅ Activated FVCD-net" && \
  which python && \
  python -V

In [ ]:
# 先升级 pip 和 setuptools
!pip install -U pip setuptools wheel



In [ ]:
# 安装不易出错的 pip-only 库
!source /opt/miniforge3/bin/activate FVCD-net && \
  pip install easydict==1.9 tensorlayer==1.11.0 --no-deps

In [ ]:
# ✅ 使用 conda 安装 TF1.14 项目常用依赖，全部为预编译版，避免 wheel 报错
!source /opt/miniforge3/bin/activate FVCD-net && \
  conda install -y \
    numpy=1.15.4 \
    scipy=1.2.0 \
    scikit-image=0.14.1 \
    imageio=2.4.1 \
    matplotlib=3.0.3 \
    -c conda-forge && \
  echo "✅ 所有核心库已使用 conda 安装完成（避免 pip build 错误）"


In [ ]:
# ⚠️ 删除项目目录中的旧 tensorlayer 源码文件夹
!rm -rf /notebooks/Code/DL_net/tensorlayer

In [ ]:
# 激活环境并手动安装所有 TensorLayer 1.11.0 明确要求的版本
!source /opt/miniforge3/bin/activate FVCD-net && \
  pip install \
    lxml==4.2.6 \
    requests==2.19.1 \
    scikit-learn==0.20.4 \
    tqdm==4.27.0 \
    progressbar2==3.38.0 \
    scipy==1.1.0 \
    wrapt==1.10.11

In [ ]:
# 强制安装兼容 TensorLayer 1.11.0 的 lxml，并覆盖系统路径版本
!source /opt/miniforge3/bin/activate FVCD-net && \
  pip install --force-reinstall --no-cache-dir lxml==4.2.6


In [ ]:
import sys
!{sys.executable} -m pip install numpy==1.15.4 tifffile scikit-image

In [ ]:
import tensorlayer as tl
print("📂 TensorLayer 来自路径：", tl.__file__)
print("🔢 版本：", tl.__version__)

In [ ]:
import os
os.chdir("/notebooks/Code/DL_net")


In [ ]:
!source /opt/miniforge3/bin/activate FVCD-net && python /notebooks/Code/DL_net/convert_tif2npy.py

In [ ]:
import os
import numpy as np
import imageio

# ✅ 原始 Stack 文件夹路径
stack_folder = '/notebooks/Code/DL_net/data_npy/Mito_View3_[LF160_SF160_WF480]/Stack'

# ✅ 新的归一化后保存文件夹
normalized_folder = '/notebooks/Code/DL_net/data_npy/Mito_View3_[LF160_SF160_WF480]/Stack_normalized'
os.makedirs(normalized_folder, exist_ok=True)

file_list = sorted([f for f in os.listdir(stack_folder) if f.endswith('.npy')])

print(f"共找到 {len(file_list)} 个Stack文件，开始归一化处理...")

for idx, file_name in enumerate(file_list):
    file_path = os.path.join(stack_folder, file_name)
    save_path = os.path.join(normalized_folder, file_name)

    # 读取npy
    stack = np.load(file_path)

    # 防止出现极小值异常，增加健壮性
    if np.max(stack) > 0:
        stack = stack.astype(np.float32)
        stack = stack / np.max(stack)
    else:
        print(f"⚠️ 警告: {file_name} 最大值为0，跳过归一化。")
        continue

    # 保存到新的文件夹
    np.save(save_path, stack)

    if idx % 10 == 0 or idx == len(file_list)-1:
        print(f"✅ 已处理 {idx+1}/{len(file_list)} 个文件: {file_name}")

print("🎯 所有Stack文件归一化处理完成并保存到新路径！")

In [ ]:
import os
import numpy as np

# 你的 Stack_normalized 路径
stack_path = '/notebooks/Code/DL_net/data_npy/Mito_View3_[LF160_SF160_WF480]/Stack_normalized/'

# 随便取10个文件检测
npy_files = sorted([f for f in os.listdir(stack_path) if f.endswith('.npy')])[:10]

for filename in npy_files:
    filepath = os.path.join(stack_path, filename)
    arr = np.load(filepath)

    print(f"📂 文件: {filename}")
    print(f"✅ max值: {np.max(arr):.6f}, min值: {np.min(arr):.6f}, mean值: {np.mean(arr):.6f}")
    print("-" * 50)

In [ ]:
!/opt/miniforge3/envs/FVCD-net/bin/python -u /notebooks/Code/DL_net/train.py 2>&1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 假设你在 /notebooks/Code/DL_net/data_npy/Mito_View3_[LF160_SF160_WF480]/LR/ 下面有npy文件
lfp_sample = np.load('/notebooks/Code/DL_net/data_npy/Mito_View3_[LF160_SF160_WF480]/LR/Data001_idx0002.npy')
hr_sample = np.load('/notebooks/Code/DL_net/data_npy/Mito_View3_[LF160_SF160_WF480]/HR/Data001_idx0002.npy')
stack_sample = np.load('/notebooks/Code/DL_net/data_npy/Mito_View3_[LF160_SF160_WF480]/Stack_normalized/Data001_idx0002.npy')

plt.figure(figsize=(15,5))
plt.subplot(1,3,1)
plt.title('LFP Sample')
plt.imshow(lfp_sample[...,0], cmap='gray')
plt.subplot(1,3,2)
plt.title('HR Sample')
plt.imshow(hr_sample[...,0], cmap='gray')
plt.subplot(1,3,3)
plt.title('Stack Sample')
plt.imshow(stack_sample[...,0], cmap='gray')
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# 指向归一化后的Stack_npy目录
stack_folder = '/notebooks/Code/DL_net/data_npy/Mito_View3_[LF160_SF160_WF480]/Stack_normalized/'

# 读取一个样本
file_list = sorted([f for f in os.listdir(stack_folder) if f.endswith('.npy')])
sample_file = os.path.join(stack_folder, file_list[0])

stack = np.load(sample_file)

# 打印最大最小值确认
print(f"归一化后的stack min: {stack.min()}, max: {stack.max()}")

# 显示若干个slice
for i in range(0, stack.shape[-1], 5):  # 每隔5张看一张
    plt.figure(figsize=(6,6))
    plt.imshow(stack[...,i], cmap='gray')
    plt.title(f'Normalized Stack - Slice {i}')
    plt.colorbar()
    plt.show()


In [ ]:
import os
os.chdir('/notebooks/Code/DL_net')


In [ ]:
conda install -c conda-forge scikit-image

In [ ]:
conda install -c menpo opencv

In [ ]:
# 步骤1: 运行SR阶段
!source /opt/miniforge3/bin/activate FVCD-net && python /notebooks/Code/DL_net/sr_stage.py --ckpt=110

In [ ]:
# 步骤2: 运行Recon阶段
!source /opt/miniforge3/bin/activate FVCD-net && python /notebooks/Code/DL_net/recon_stage.py --ckpt=110

In [ ]:
#保存结果--✅ Step 1：压缩推理输出 .tif 文件夹为 zip
import shutil
import os

# 源目录
src_folder = "/notebooks/Code/DL_net/data_npy/Mito_View3_[LF160_SF160_WF480]/LR/SR_[Mito]_view3_L5c126b8_reallasttry"

# 输出压缩文件名（zip）
zip_file = src_folder + ".zip"

# 如果存在旧压缩包先删除
if os.path.exists(zip_file):
    os.remove(zip_file)

# 创建 zip 压缩包
shutil.make_archive(base_name=zip_file.replace(".zip", ""), format='zip', root_dir=src_folder)

print(f"✅ 压缩完成，已保存为: {zip_file}")

In [ ]:
conda install pandas openpyxl -y

In [ ]:
conda install numpy matplotlib seaborn scikit-image pillow -y

In [ ]:
# 导入模块
import os
import sys
import zipfile
from datetime import datetime
sys.path.append('/notebooks/Code/DL_net')  # 确保脚本所在目录在Python路径中
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import FileLink, display

# 定义自定义函数(如果model_evaluation模块加载有问题)
def load_validation_results(metrics_file):
    """从验证指标文件加载结果"""
    results = []
    with open(metrics_file, 'r') as f:
        lines = f.readlines()
    
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 3:  # 确保行有足够的数据
            filename = parts[0]
            # 正确提取PSNR和SSIM值
            psnr_part = [p for p in parts if 'PSNR:' in p]
            ssim_part = [p for p in parts if 'SSIM:' in p]
            
            if psnr_part and ssim_part:
                psnr = float(psnr_part[0].split(':')[1])
                ssim = float(ssim_part[0].split(':')[1])
                results.append({'filename': filename, 'psnr': psnr, 'ssim': ssim})
    
    df = pd.DataFrame(results)
    return df

# 尝试导入自定义模块，如果失败则使用上面定义的函数
try:
    from model_evaluation import load_validation_results, analyze_performance_statistics
    from model_analysis import analyze_performance_by_group
except ImportError:
    print("无法导入自定义模块，使用内联定义的函数")
    
    def analyze_performance_statistics(df):
        """分析性能统计数据并生成图表"""
        # 计算总体统计信息
        stats = {
            "总样本数": len(df),
            "平均PSNR": df['psnr'].mean(),
            "平均SSIM": df['ssim'].mean(),
            "PSNR范围": f"{df['psnr'].min():.2f} - {df['psnr'].max():.2f}",
            "SSIM范围": f"{df['ssim'].min():.4f} - {df['ssim'].max():.4f}",
            "PSNR标准差": df['psnr'].std(),
            "SSIM标准差": df['ssim'].std()
        }
        
        # 输出统计信息
        for key, value in stats.items():
            print(f"{key}: {value}")
        
        # 创建性能分布直方图
        plt.figure(figsize=(16, 6))
        
        plt.subplot(1, 2, 1)
        sns.histplot(df['psnr'], kde=True, bins=20)
        plt.title('PSNR分布')
        plt.xlabel('PSNR (dB)')
        plt.ylabel('样本数量')
        plt.axvline(df['psnr'].mean(), color='red', linestyle='--', 
                    label=f'平均值: {df["psnr"].mean():.2f}')
        plt.legend()
        
        plt.subplot(1, 2, 2)
        sns.histplot(df['ssim'], kde=True, bins=20)
        plt.title('SSIM分布')
        plt.xlabel('SSIM')
        plt.ylabel('样本数量')
        plt.axvline(df['ssim'].mean(), color='red', linestyle='--', 
                    label=f'平均值: {df["ssim"].mean():.4f}')
        plt.legend()
        
        plt.tight_layout()
        return stats

    def analyze_performance_by_group(results_df, group_by='image_type'):
        """按图像类型分析性能表现"""
        # 添加图像分类标签
        if group_by not in results_df.columns:
            results_df[group_by] = results_df['filename'].apply(
                lambda x: x.split('_')[0] if '_' in x else x.split('.')[0]
            )
        
        # 按图像类型分组分析性能
        group_performance = results_df.groupby(group_by).agg({
            'psnr': ['mean', 'std', 'min', 'max'],
            'ssim': ['mean', 'std', 'min', 'max']
        }).round(4)
        
        print("不同类型图像的性能:")
        print(group_performance)
        
        # 可视化不同类型图像的性能
        plt.figure(figsize=(14, 6))
        sns.boxplot(x=group_by, y='psnr', data=results_df)
        plt.title(f'不同{group_by}的PSNR分布')
        plt.xlabel(group_by)
        plt.ylabel('PSNR (dB)')
        plt.xticks(rotation=45 if len(results_df[group_by].unique()) > 10 else 0)
        plt.tight_layout()
        
        plt.figure(figsize=(14, 6))
        sns.boxplot(x=group_by, y='ssim', data=results_df)
        plt.title(f'不同{group_by}的SSIM分布')
        plt.xlabel(group_by)
        plt.ylabel('SSIM')
        plt.xticks(rotation=45 if len(results_df[group_by].unique()) > 10 else 0)
        plt.tight_layout()
        
        # 找出每种类型中表现最好和最差的样本
        best_worst_by_type = {}
        for img_type in results_df[group_by].unique():
            type_df = results_df[results_df[group_by] == img_type]
            best = type_df.nlargest(1, 'psnr')
            worst = type_df.nsmallest(1, 'psnr')
            
            print(f"\n类型 {img_type} 表现最好的样本: {best['filename'].values[0]}, "
                f"PSNR={best['psnr'].values[0]:.2f}, SSIM={best['ssim'].values[0]:.4f}")
            print(f"类型 {img_type} 表现最差的样本: {worst['filename'].values[0]}, "
                f"PSNR={worst['psnr'].values[0]:.2f}, SSIM={worst['ssim'].values[0]:.4f}")
            
            best_worst_by_type[img_type] = {'best': best, 'worst': worst}
        
        return group_performance, best_worst_by_type

# 创建时间戳和用户标识的结果目录
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
username = 'Placebo303'  # 用户名
result_dir_name = f'FVCD_analysis_{username}_{timestamp}'
results_dir = os.path.join('/notebooks/Code/DL_net', result_dir_name)
os.makedirs(results_dir, exist_ok=True)

# 正确的文件路径
metrics_file = '/notebooks/Code/DL_net/data_npy/Mito_View3_[LF160_SF160_WF480]/LR/SR_[Mito]_view3_L5c126b8_reallasttry/validation_metrics.txt'

# 1. 加载验证结果并分析
print("正在加载和分析验证结果...")
results_df = load_validation_results(metrics_file)
stats = analyze_performance_statistics(results_df)

# 保存性能分布图
plt.savefig(os.path.join(results_dir, 'performance_distribution.png'), dpi=300)
plt.close()

# 2. 按图像类型分析性能
print("\n正在按图像类型进行性能分析...")
group_stats, best_worst = analyze_performance_by_group(results_df)

# 保存按类型分析的图表
plt.figure(figsize=(14, 6))
sns.boxplot(x='image_type', y='psnr', data=results_df)
plt.title(f'不同image_type的PSNR分布')
plt.xlabel('Image Type')
plt.ylabel('PSNR (dB)')
plt.xticks(rotation=45 if len(results_df['image_type'].unique()) > 10 else 0)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'psnr_by_type.png'), dpi=300)
plt.close()

plt.figure(figsize=(14, 6))
sns.boxplot(x='image_type', y='ssim', data=results_df)
plt.title(f'不同image_type的SSIM分布')
plt.xlabel('Image Type')
plt.ylabel('SSIM')
plt.xticks(rotation=45 if len(results_df['image_type'].unique()) > 10 else 0)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'ssim_by_type.png'), dpi=300)
plt.close()

# 3. 保存各种分析结果
print("\n正在保存分析结果...")

# 保存DataFrame到Excel和CSV
results_df.to_excel(os.path.join(results_dir, 'validation_results.xlsx'), index=False)
results_df.to_csv(os.path.join(results_dir, 'validation_results.csv'), index=False)

# 保存统计摘要
with open(os.path.join(results_dir, 'stats_summary.txt'), 'w') as f:
    f.write(f"FVCD-net 验证结果分析 - {timestamp}\n")
    f.write(f"用户: {username}\n\n")
    f.write("=== 总体性能统计 ===\n")
    for key, value in stats.items():
        f.write(f"{key}: {value}\n")
    
    f.write("\n=== 按类型统计性能 ===\n")
    f.write(group_stats.to_string())

# 保存详细分析报告
with open(os.path.join(results_dir, 'detailed_analysis.md'), 'w') as f:
    f.write(f"# FVCD-net 验证结果分析报告\n\n")
    f.write(f"生成时间: {timestamp}\n")
    f.write(f"分析用户: {username}\n\n")
    
    f.write("## 总体性能评估\n\n")
    f.write("### 主要指标统计\n")
    f.write(f"- **平均PSNR**: {stats['平均PSNR']:.2f} dB\n")
    f.write(f"- **平均SSIM**: {stats['平均SSIM']:.4f}\n")
    f.write(f"- **PSNR范围**: {stats['PSNR范围']} dB\n")
    f.write(f"- **SSIM范围**: {stats['SSIM范围']}\n")
    
    # 性能分布分析
    psnr_values = results_df['psnr'].values
    high_quality = sum(psnr_values > 40.0) / len(psnr_values) * 100
    medium_quality = sum((psnr_values <= 40.0) & (psnr_values > 30.0)) / len(psnr_values) * 100
    low_quality = sum(psnr_values <= 30.0) / len(psnr_values) * 100
    
    f.write("\n### 性能分布\n")
    f.write(f"- **优秀重建质量** (PSNR > 40dB): {high_quality:.1f}% 的样本\n")
    f.write(f"- **良好重建质量** (30-40dB): {medium_quality:.1f}% 的样本\n")
    f.write(f"- **需要改进** (< 30dB): {low_quality:.1f}% 的样本\n\n")
    
    f.write("## 按类型分析\n\n")
    f.write("不同数据类型的平均性能如下：\n\n")
    f.write("| 类型 | 平均PSNR | 标准差 | 最小值 | 最大值 | 平均SSIM |\n")
    f.write("|------|---------|-------|-------|-------|--------|\n")
    
    for idx in group_stats.index:
        f.write(f"| {idx} | {group_stats.loc[idx][('psnr', 'mean')]:.2f} | ")
        f.write(f"{group_stats.loc[idx][('psnr', 'std')]:.2f} | ")
        f.write(f"{group_stats.loc[idx][('psnr', 'min')]:.2f} | ")
        f.write(f"{group_stats.loc[idx][('psnr', 'max')]:.2f} | ")
        f.write(f"{group_stats.loc[idx][('ssim', 'mean')]:.4f} |\n")
    
    f.write("\n## 性能异常分析\n\n")
    f.write("### 性能最佳的样本\n")
    best_samples = results_df.nlargest(5, 'psnr')
    for i, (_, row) in enumerate(best_samples.iterrows()):
        f.write(f"{i+1}. {row['filename']} - PSNR: {row['psnr']:.2f}, SSIM: {row['ssim']:.4f}\n")
    
    f.write("\n### 性能最差的样本\n")
    worst_samples = results_df.nsmallest(5, 'psnr')
    for i, (_, row) in enumerate(worst_samples.iterrows()):
        f.write(f"{i+1}. {row['filename']} - PSNR: {row['psnr']:.2f}, SSIM: {row['ssim']:.4f}\n")

# 创建ZIP文件
zip_path = os.path.join('/notebooks/Code/DL_net', f'FVCD_analysis_results_{username}_{timestamp}.zip')
print(f"\n正在创建ZIP文件: {zip_path}...")

with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk(results_dir):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, os.path.relpath(file_path, os.path.dirname(results_dir)))

# 创建下载链接
print("\n分析完成! 点击下面的链接下载完整的分析结果:")
display(FileLink(zip_path))
print(f"\n所有分析结果已打包至: {zip_path}")
print(f"所有单独文件也可在此目录查看: {results_dir}")